# Daily Challenge : Analyse de texte (Word Cloud) — Lewis Carroll

**Objectifs**
- Prétraitement de texte (nettoyage, tokenisation, stopwords, stemming, lemmatisation)
- Analyse de texte : POS tagging, reconnaissance d'entités nommées (NER)
- Bag of Words (BoW)
- TF-IDF

**Corpus** : trois livres de Lewis Carroll récupérés depuis le Projet Gutenberg :
1. *Alice's Adventures in Wonderland*
2. *Through the Looking-Glass, and What Alice Found There*
3. *A Tangled Tale*

> ⚠️ Important : ce notebook est pensé pour être exécuté dans un environnement virtuel dédié au cours de NLP (`python -m venv nlp_env` puis activation avant `pip install`).


In [ ]:
# Installation des paquets nécessaires (à décommenter si besoin, notamment sur Colab)
# !pip install requests nltk spacy scikit-learn wordcloud matplotlib pandas
# !python -m spacy download en_core_web_sm


In [ ]:
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import spacy

from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Ressources NLTK nécessaires
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

# Modèle spaCy pré-entraîné
nlp = spacy.load('en_core_web_sm')


## 1. Prétraitement du texte

In [ ]:
URLS = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",      # Alice's Adventures in Wonderland
    "https://www.gutenberg.org/cache/epub/12/pg12.txt",      # Through the Looking-Glass
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt" # A Tangled Tale
]

TITLES = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale"
]


In [ ]:
def clean_text(text):
    """Nettoie un texte brut : supprime tout ce qui n'est pas un mot
    (chiffres, symboles, ponctuation, retours à la ligne multiples, etc.)
    et normalise les espaces."""
    # On remplace tout caractère qui n'est pas une lettre par un espace
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    # On normalise les espaces multiples
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_texts(urls):
    """Reçoit une liste d'URLs, télécharge chaque texte, le nettoie
    (suppression des caractères non alphabétiques via regex) et retourne
    la liste des textes nettoyés (le corpus)."""
    corpus = []
    for url in urls:
        response = requests.get(url)
        response.raise_for_status()
        raw_text = response.text
        cleaned = clean_text(raw_text)
        corpus.append(cleaned)
    return corpus


corpus = load_texts(URLS)
print(f"Nombre de textes chargés : {len(corpus)}")


### 2. Aperçu des 200 premiers caractères de chaque texte

In [ ]:
for title, text in zip(TITLES, corpus):
    print(f"--- {title} ---")
    print(text[:200])
    print()


**Observation** : les 200 premiers caractères correspondent à l'en-tête légale
du Projet Gutenberg (titre, mentions de licence, informations sur le fichier...)
et non au contenu réel du livre. De la même façon, la fin du fichier contient
un long texte de licence ("END OF THE PROJECT GUTENBERG EBOOK ..."). Ces parties
ne sont pas pertinentes pour l'analyse et doivent être supprimées avant de
poursuivre.

On utilise donc les repères `START` et `END` (présents dans les balises
`*** START OF THE PROJECT GUTENBERG EBOOK ... ***` et
`*** END OF THE PROJECT GUTENBERG EBOOK ... ***`) pour ne garder que le texte
utile via du slicing.


In [ ]:
def trim_boilerplate(text):
    """Supprime l'en-tête et le pied de page du Projet Gutenberg en se
    basant sur les mots 'START' et 'END' (le texte ayant déjà été nettoyé,
    les balises *** ont disparu mais les mots START/END subsistent)."""
    start_idx = text.find(" START")
    end_idx = text.find(" END")

    if start_idx != -1:
        # On saute jusqu'à la fin de la ligne d'annonce du START
        start_idx = text.find(" ", start_idx + 1, start_idx + 200)
        text = text[start_idx:]

    end_idx = text.find(" END")
    if end_idx != -1:
        text = text[:end_idx]

    return text.strip()


corpus_trimmed = [trim_boilerplate(text) for text in corpus]

for title, text in zip(TITLES, corpus_trimmed):
    print(f"--- {title} ---")
    print(text[:200])
    print("...")
    print(text[-200:])
    print()


### 3. Tokenisation

In [ ]:
tokens_per_book = []

for title, text in zip(TITLES, corpus_trimmed):
    tokens = word_tokenize(text.lower())
    tokens_per_book.append(tokens)
    print(f"--- {title} ---")
    print(tokens[:150])
    print()


### 4. Suppression des stopwords

In [ ]:
stop_words = set(stopwords.words('english'))

# Quelques stopwords que l'on s'attend à trouver avant nettoyage
sample_stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']

print("Occurrences AVANT suppression des stopwords :")
for title, tokens in zip(TITLES, tokens_per_book):
    counts = {w: tokens.count(w) for w in sample_stopwords}
    print(f"{title}: {counts}")

tokens_no_stop_per_book = []
for tokens in tokens_per_book:
    filtered = [t for t in tokens if t not in stop_words]
    tokens_no_stop_per_book.append(filtered)

print("\nOccurrences APRÈS suppression des stopwords :")
for title, tokens in zip(TITLES, tokens_no_stop_per_book):
    counts = {w: tokens.count(w) for w in sample_stopwords}
    print(f"{title}: {counts}")


### 5. Stemming avec PorterStemmer

In [ ]:
stemmer = PorterStemmer()
stemmed_tokens_per_book = []

for title, tokens in zip(TITLES, tokens_no_stop_per_book):
    stemmed = [stemmer.stem(t) for t in tokens]
    stemmed_tokens_per_book.append(stemmed)
    print(f"--- {title} ---")
    print(stemmed[:50])
    print()


### 6. Lemmatisation avec spaCy

In [ ]:
lemmatized_tokens_per_book = []

for title, tokens in zip(TITLES, tokens_no_stop_per_book):
    # On reconstruit une chaîne à partir des tokens filtrés pour la passer à spaCy
    doc = nlp(" ".join(tokens))
    lemmas = [token.lemma_ for token in doc]
    lemmatized_tokens_per_book.append(lemmas)
    print(f"--- {title} ---")
    print(lemmas[:50])
    print()


### 7. Stemming vs Lemmatisation : analyse

- Le **stemming** (`PorterStemmer`) applique des règles heuristiques pour
  tronquer les mots à leur "racine" approximative. Le résultat n'est pas
  toujours un mot valide de la langue anglaise (ex : `"having"` → `"have"`,
  mais aussi des cas comme `"studies"` → `"studi"`). C'est une méthode rapide
  mais moins précise.
- La **lemmatisation** (spaCy) s'appuie sur un modèle linguistique et un
  dictionnaire pour renvoyer le **lemme réel** du mot, c'est-à-dire sa forme
  canonique telle qu'on la trouverait dans un dictionnaire (ex : `"having"` →
  `"have"`, `"mice"` → `"mouse"`). Elle est plus lente à calculer mais donne
  des résultats plus propres et exploitables linguistiquement.

En résumé : la lemmatisation est plus précise car elle prend en compte le
contexte grammatical et le vocabulaire réel, alors que le stemming se contente
de couper des suffixes selon des règles fixes.


### 8. POS Tagging (NLTK)

In [ ]:
pos_tags_per_book = []

for title, tokens in zip(TITLES, tokens_no_stop_per_book):
    tags = nltk.pos_tag(tokens)
    pos_tags_per_book.append(tags)
    print(f"--- {title} ---")
    print(tags[:50])
    print()


### 9. Reconnaissance d'entités nommées (NER) avec NLTK

In [ ]:
from nltk import ne_chunk

for title, tags in zip(TITLES, pos_tags_per_book):
    print(f"--- {title} ---")
    tree = ne_chunk(tags[:300])  # limité pour la lisibilité de l'affichage
    entities = [
        (" ".join(c[0] for c in chunk), chunk.label())
        for chunk in tree
        if hasattr(chunk, 'label')
    ]
    print(entities[:20])
    print()


## Analyse du texte

### 1. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title, tokens in zip(axes, TITLES, tokens_no_stop_per_book):
    text_for_cloud = " ".join(tokens)
    wc = WordCloud(width=600, height=400, background_color='white').generate(text_for_cloud)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()


### 2. Bag of Words (BoW)

Pour le BoW, on utilise le texte **lemmatisé** (sans stopwords) : c'est la
version qui réduit le plus le bruit (mots à la forme canonique) tout en
restant lisible, ce qui donne le comptage le plus pertinent des mots réels.


In [ ]:
# On reconstruit un document (une chaîne) par livre à partir des lemmes
documents_lemmatized = [" ".join(lemmas) for lemmas in lemmatized_tokens_per_book]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(documents_lemmatized)
feature_names = count_vectorizer.get_feature_names_out()

bow_array = bow_matrix.toarray()

top5_bow_per_book = []
for i, title in enumerate(TITLES):
    counts = bow_array[i]
    top5_idx = counts.argsort()[::-1][:5]
    top5_words = [(feature_names[idx], counts[idx]) for idx in top5_idx]
    top5_bow_per_book.append(top5_words)
    print(f"--- {title} ---")
    print(top5_words)
    print()


### 3. Lecture de la matrice BoW

- **Document** : chaque ligne de `bow_matrix` correspond à un document (ici,
  un livre) — le numéro de document est donc l'indice de la ligne (0, 1, 2).
- **Index** : chaque colonne correspond à un mot du vocabulaire ; l'indice de
  colonne correspond à la position du mot dans `feature_names`
  (`count_vectorizer.get_feature_names_out()`).
- **Valeur** : la valeur à l'intersection `(document, index)` indique **combien
  de fois** ce mot apparaît dans ce document.


In [ ]:
print("Forme de la matrice BoW (documents x vocabulaire) :", bow_matrix.shape)
print()
print("Exemple de représentation creuse (document, index) -> compte :")
print(bow_matrix[0])  # affichage de la représentation creuse du 1er document


### 4. Diagrammes circulaires (pie plots) des 5 mots les plus fréquents (BoW)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title, top5 in zip(axes, TITLES, top5_bow_per_book):
    words = [w for w, c in top5]
    counts = [c for w, c in top5]
    labels = [f"{w} ({c})" for w, c in top5]
    ax.pie(counts, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(title)

plt.tight_layout()
plt.show()


### 5. Analyse : ces mots sont-ils informatifs ?

Les mots les plus fréquents obtenus via le BoW brut sont généralement des mots
attendus et peu informatifs pour caractériser chaque livre spécifiquement :
des mots comme `"said"`, `"alice"`, `"one"`, `"little"` reviennent très
souvent simplement parce que le livre parle constamment du personnage
principal et utilise un style narratif répétitif. Ce ne sont pas des mots qui
permettent de distinguer le contenu ou le thème propre à chaque livre — d'où
l'intérêt d'utiliser le TF-IDF pour pondérer ces mots trop fréquents.


## Résoudre le problème de fréquence avec TF-IDF

Le TF-IDF pondère chaque mot en fonction de sa fréquence dans un document
relativement à sa fréquence dans l'ensemble du corpus. Un mot très fréquent
dans un document mais rare dans les autres documents aura un score élevé
(il est donc discriminant pour ce document). Un mot fréquent dans tous les
documents (ex: "said", "alice") aura un score plus faible.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents_lemmatized)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_array = tfidf_matrix.toarray()

top5_tfidf_per_book = []
for i, title in enumerate(TITLES):
    scores = tfidf_array[i]
    top5_idx = scores.argsort()[::-1][:5]
    top5_words = [(tfidf_feature_names[idx], round(scores[idx], 4)) for idx in top5_idx]
    top5_tfidf_per_book.append(top5_words)
    print(f"--- {title} ---")
    print(top5_words)
    print()


### 2. Diagrammes circulaires des 5 mots les plus pertinents (TF-IDF)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, title, top5 in zip(axes, TITLES, top5_tfidf_per_book):
    words = [w for w, c in top5]
    scores = [c for w, c in top5]
    labels = [f"{w} ({c})" for w, c in top5]
    ax.pie(scores, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title(title)

plt.tight_layout()
plt.show()


**Comparaison** : contrairement au BoW brut, les mots ressortant du TF-IDF
sont beaucoup plus spécifiques à chaque livre (des noms de personnages ou de
concepts propres à chaque histoire plutôt que des mots génériques comme
"said" ou "alice"), ce qui les rend plus informatifs pour caractériser le
contenu distinctif de chaque texte.
